In [ ]:
import sys

sys.path.append("../..")
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from interpreto.attributions import Saliency
from interpreto.concepts.metrics import AttrSim
from interpreto.model_wrapping.llm_interface import HuggingFaceLLM

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "textattack/distilbert-base-uncased-ag-news"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
dataset = load_dataset("fancyzhx/ag_news")

n_train = 700
train_inputs = dataset["train"]["text"][:n_train]
classes_names = dataset["train"].features["label"].names

In [ ]:
def predict_in_batches(model, tokenizer, texts, device, batch_size=32):
    model.eval()
    predictions = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]

            encoded = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True)
            encoded = {k: v.to(device) for k, v in encoded.items()}

            outputs = model(**encoded)
            preds = outputs.logits.argmax(dim=-1).cpu()
            predictions.append(preds)

            del encoded, outputs, preds
            torch.cuda.empty_cache()

    return torch.cat(predictions, dim=0)


train_predictions = predict_in_batches(model, tokenizer, train_inputs, device, batch_size=32)

In [8]:
attrsim = AttrSim(classes=classes_names)
train_labels = torch.tensor(dataset["train"]["label"][:n_train])

# Select a balanced pool (good/miss) then force 10 good + 10 miss
# for the ConSim learning phase.
indices, samples, selected_labels, selected_predictions = attrsim.select_examples(
    inputs=train_inputs,
    labels=train_labels,
    predictions=train_predictions,
    nb_samples=30,
    seed=0,
)

good_mask = selected_labels == selected_predictions
good_idx = torch.where(good_mask)[0]
miss_idx = torch.where(~good_mask)[0]

lp_good = good_idx[:10]
lp_miss = miss_idx[:10]
lp_idx = torch.cat([lp_good, lp_miss])

all_idx = torch.arange(len(samples))
ep_idx = all_idx[~torch.isin(all_idx, lp_idx)]
ordered_idx = torch.cat([lp_idx, ep_idx])

samples = [samples[i] for i in ordered_idx.tolist()]
selected_labels = selected_labels[ordered_idx]
selected_predictions = selected_predictions[ordered_idx]
indices = indices[ordered_idx]


In [ ]:
# AttrSim evaluation (same selected samples, but with token attributions as explanations)

saliency = Saliency(
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

# Build attributions on the selected samples for each sample predicted class.
attr_outputs = saliency.explain(
    samples,
    targets=selected_predictions,
)

attr_system_prompt, attr_user_prompts, attr_model_predictions = attrsim.construct_prompt(
    setting=AttrSim.prompt_types.E1_attribution_with_lp,
    interesting_samples=samples,
    corresponding_predictions=selected_predictions,
    corresponding_labels=selected_labels,
    nb_learning_samples=20,
    corresponding_attribution=attr_outputs,
)

IndexError: index 2 is out of bounds for dimension 0 with size 1

In [ ]:
llm = HuggingFaceLLM(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    batch_size=2,
    device=device,
)


attr_responses = llm.batch_generate(
    attr_system_prompt,
    attr_user_prompts,
    max_new_tokens=16,
    do_sample=False,
)

attr_score = attrsim.score_from_responses(attr_responses, attr_model_predictions)

print("AttrSim score:", attr_score)
print("AttrSim responses preview:", attr_responses[:2])